# 06 - Can the explanations be trusted?

Step 6 of `THESIS_GUIDE.md` (PIANO.md C1). The models predict a weak signal (notebook 03), so their explanations may be as noisy as their predictions. Before comparing explanations across models (notebook 07), each method is checked on one run per model: DJI, T = 20, L = 5, seed 42, test year 2020, on the first explained days.

1. **Sanity check** (Adebayo et al. 2018): the attributions of the trained model against those of the same architecture with the initial weights of its seed (where training started). Similar attributions mean the method shows the structure of the inputs or of the architecture rather than what the model learned.
2. **Faithfulness**: deletion curves. Each stock's inputs are set to the baseline (the day's average stock) in the order of their attribution, most positive first; a faithful attribution makes the score fall fast, compared with a random order. Two modes: all stocks of the day at once (`joint`, as IG's joint path) and one stock at a time (`own`). AOPC: the mean fall over the steps.
3. **Agreement between methods** on the family ranking: IG (gross and net shares, `xai/aggregate.py`), Saliency, exact Shapley values and LIME on families (`xai/shap_lime.py`), and the two permutation measures.
4. **The IG settings**: joint against own path, day-mean against zero baseline.

The choice of the methods and of the family measure used in notebook 07 and in the thesis follows from these results.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import torch
from IPython.display import display
from scipy.stats import spearmanr

from xaifin import viz
from xaifin.analysis.agreement import agreement, agreement_matrix
from xaifin.config import RESULTS_ROOT, TOP_K
from xaifin.models.registry import load_run
from xaifin.xai.aggregate import global_importance, input_table
from xaifin.xai.base import explain_dates, explain_run, load_attributions, xai_dir
from xaifin.xai.evaluation import aopc, deletion_curve, sanity_check, untrained_copy
from xaifin.xai.gradient import IntegratedGradients, Saliency
from xaifin.xai.permutation import FamilyPermutation, summarize
from xaifin.xai.shap_lime import FamilyLIME, FamilyShapley, family_global

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
UNIVERSE, SEED, YEAR, N_DAYS = "dji", 42, 2020, 5
K = TOP_K[UNIVERSE]
MODELS = viz.MODEL_ORDER
runs = {m: RESULTS_ROOT / f"Regression/{m}/{UNIVERSE}/sl20_pl5/seed{SEED}/y{YEAR}" for m in MODELS}
runs = {m: p for m, p in runs.items() if (p / "model.pth").exists()}
adapters = {m: load_run(p) for m, p in runs.items()}
dates = {m: explain_dates(a) for m, a in adapters.items()}
print(f"{UNIVERSE} seed {SEED} y{YEAR}: models {list(adapters)}; {N_DAYS} days per check")

def days(model, n=N_DAYS):
    wanted = set(dates[model][:n])
    return [b for b in adapters[model].day_batches("test") if b.date in wanted]

## 1. Sanity check: trained against untrained

Per model: how far training moved the weights from their initial values, how similar the scores of the two models are, and how similar their IG attributions are (median over stock-days of the Spearman correlation of |attribution| per feature and per input). The last column compares IG with the plain deviation of the input from the day's average stock, |x - b|, the part of IG that does not depend on the model.

In [ ]:
rows = []
for model, a in adapters.items():
    u = untrained_copy(a)
    w1 = torch.cat([p.flatten() for p in a.model.parameters()])
    w0 = torch.cat([p.flatten() for p in u.model.parameters()])
    s = sanity_check(a, IntegratedGradients(), dates[model][:N_DAYS], "pred", K)
    score_rho, input_rho = [], []
    for b in days(model):
        score_rho.append(spearmanr(a.predict(b), u.predict(b))[0])
        ig = IntegratedGradients().explain(a, b, "pred", K)
        x = b.x.numpy()
        dev = np.abs(x - x.mean(0))
        input_rho.append(np.median([spearmanr(np.abs(ig[i]).ravel(), dev[i].ravel())[0] for i in range(len(x))]))
    rows.append({"model": model, "weight change": float((w1 - w0).norm() / w0.norm()), "scores trained vs untrained": np.mean(score_rho),
                 "IG trained vs untrained (features)": s["feature"].median(), "IG trained vs untrained (inputs)": s["input"].median(),
                 "IG vs |x - b|": np.mean(input_rho)})
sanity = pd.DataFrame(rows).set_index("model")
sanity

## 2. Faithfulness: deletion curves

Fall of the target (the score) when a growing share of each stock's inputs is set to the baseline, in the order of IG or Saliency (most positive first) or at random, averaged over stocks and days.

In [ ]:
curves = []
for model, a in adapters.items():
    for b in days(model):
        for name, explainer in (("IG", IntegratedGradients()), ("Saliency", Saliency())):
            attr = explainer.explain(a, b, "pred", K)
            for mode in ("joint", "own"):
                curves.append(deletion_curve(a, b, attr, "pred", K, mode=mode).assign(model=model, method=name, mode=mode))
curves = pd.concat(curves)
aopc_table = curves.groupby(["model", "mode", "method", "order"])["drop"].mean().unstack(["method", "order"])
display(aopc_table)  # AOPC: mean fall of the score over the deletion steps; "random" is the reference
own = curves[(curves["mode"] == "own")]
plot = own.groupby(["model", "method", "order", "fraction"])["drop"].mean().reset_index()
plot["curve"] = np.where(plot["order"] == "random", "random", plot["method"])
plot = plot.drop_duplicates(["model", "curve", "fraction"])
px.line(plot, x="fraction", y="drop", color="curve", facet_col="model", facet_col_wrap=3, markers=True,
        title="Deletion curves, one stock at a time: fall of the score as the most attributed inputs are removed").show()

## 3. Do the methods agree?

On every explained day of the run: family importance from IG (gross share, sum |a|, and net share, |sum a|), Saliency (gross), exact Shapley values and LIME on families, and permutation importance (drop in RankIC and share of the top-k replaced). Spearman correlation of the family rankings, per model; the market family is left out (zero by construction with the day-mean baseline).

In [ ]:
def group_values(adapter, explainer, target="pred"):
    out = xai_dir(explainer.name, adapter) / f"global_{target}.parquet"
    if out.exists():
        return pd.read_parquet(out)
    result = explain_run(adapter, explainer, target, K, dates[adapter.cfg.model], log=lambda m: None)
    return family_global(result["attr"], result["date"], explainer.families)

def gradient_global(adapter, method, explainer, target="pred"):
    result = load_attributions(adapter, method, target)
    if result is None:
        result = explain_run(adapter, explainer, target, K, dates[adapter.cfg.model], log=lambda m: None)
    return global_importance(result["attr"], result["date"], input_table(adapter))

fam = lambda g, value="importance": g[(g["level"] == "family") & (g["name"] != "market")].set_index("name")[value]
matrices = {}
for model, a in adapters.items():
    ig, sal = gradient_global(a, "ig_joint", IntegratedGradients()), gradient_global(a, "saliency", Saliency())
    perm_file = xai_dir("permutation", a) / "permutation.parquet"
    perm = summarize(pd.read_parquet(perm_file) if perm_file.exists() else FamilyPermutation().explain_run(a, dates[model], K))
    vectors = {"IG gross": fam(ig), "IG net": fam(ig, "net"), "Saliency": fam(sal),
               "Shapley": fam(group_values(a, FamilyShapley())), "LIME": fam(group_values(a, FamilyLIME())),
               "perm. dRankIC": perm.set_index("family")["d_RankIC"], "perm. top-k": perm.set_index("family")["topk_change"]}
    matrices[model] = agreement_matrix(vectors, "spearman")
mean_matrix = sum(matrices.values()) / len(matrices)
viz.overlap_heatmap(mean_matrix, "Agreement of the family rankings between methods (Spearman, mean over models)").show()
display(mean_matrix)
pd.DataFrame({m: {"IG net vs Shapley": mx.loc["IG net", "Shapley"], "IG gross vs Shapley": mx.loc["IG gross", "Shapley"],
                  "LIME vs Shapley": mx.loc["LIME", "Shapley"], "Shapley vs perm. top-k": mx.loc["Shapley", "perm. top-k"],
                  "Shapley vs perm. dRankIC": mx.loc["Shapley", "perm. dRankIC"]} for m, mx in matrices.items()}).T

## 4. The IG settings: path and baseline

Per stock, Spearman correlation of |attribution| per feature: joint path (used) against the own path of the guide, and day-mean baseline (used) against zero; completeness of the own path (|sum of attributions - change of the score| relative to the change).

In [ ]:
rows = []
for model, a in adapters.items():
    names = np.array(a.feature_names)
    for b in days(model, min(3, N_DAYS)):
        joint = IntegratedGradients(path="joint").explain(a, b, "pred", K)
        own = IntegratedGradients(path="own").explain(a, b, "pred", K)
        zero = IntegratedGradients(path="joint", baseline="zero").explain(a, b, "pred", K)
        for i in range(len(b.tickers)):
            per_feature = lambda v: np.abs(v[i]).sum(0)
            rows.append({"model": model, "own vs joint": spearmanr(per_feature(own), per_feature(joint))[0],
                         "zero vs mean baseline": spearmanr(per_feature(zero), per_feature(joint))[0]})
settings = pd.DataFrame(rows).groupby("model").median().reindex(MODELS)
settings

## Choice of the methods

*To be written on the final runs (seed 42, PIANO.md C1): which method is the main one, which family measure (gross or net) notebook 07 uses, and how the sanity check limits the reading of each model's explanations.*